# Georeferenced Metadata of African Bat Specimens with Taxonomic and Collection Information Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.643m-kq5a/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# To get information, we use the .metadata object attributes, as per mlcroissant best practice.
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished}\nVersion: {metadata.version}\nIdentifier: {metadata.identifier}")

## 2. Data Overview
Review available record sets, fields, columns, and their IDs.

Every entity is referenced by its `@id` for consistency.

In [ ]:
# List available record sets
record_sets = list(dataset.record_sets)
print("Record Sets (@id and name):")
for rs in record_sets:
    print(f"  - @id: {rs.id}, name: {rs.name}")

# For each record set, list its fields and columns
print("\nFields and columns in each Record Set:")
for rs in record_sets:
    print(f"\nRecord Set @id: {rs.id} (name: {rs.name})")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - @id: {field.id}, name: {field.name}, dataType: {getattr(field, 'dataType', None)}")
    print("  Columns:")
    for col in rs.columns:
        print(f"    - @id: {col.id}, name: {col.name}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Get the list of record set @ids
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = dict()

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)

# Select primary record set for specimen data extraction
# By inspecting the record set names, choose one with specimen or collection data, e.g. 'cr:SpecimenRecords' if existing.
main_record_set_id = None
for rs in dataset.record_sets:
    if 'Specimen' in (rs.name or ''):
        main_record_set_id = rs.id
        break
if main_record_set_id is None and dataframes:
    main_record_set_id = list(dataframes.keys())[0]
# Output the first few columns for exploration
if main_record_set_id:
    print(f"Main record set: {main_record_set_id}")
    print("Columns:", dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No record set with data found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Operations may include removing outliers, transforming fields, or grouping by categorical variables.

In [ ]:
import numpy as np

df = dataframes[main_record_set_id]
print(f"Available columns in the main record set:")
print(df.columns.tolist())

# Choose a numeric field for EDA (try 'cr:Latitude' or 'cr:Longitude' as likely candidates)
numeric_field_id = None
for possible in ['cr:Latitude', 'cr:Longitude', 'Latitude', 'Longitude', 'Year', 'cr:Year']:
    if possible in df.columns:
        numeric_field_id = possible
        break
if numeric_field_id is None:
    numeric_field_id = df.select_dtypes(include=[np.number]).columns[0]  # fall back
print(f"Selected numeric field: {numeric_field_id}")

# Filter records for the numeric field (e.g. Latitude > 0)
if df[numeric_field_id].dtype.kind in 'O':
    # Try converting to numeric in case it's string (object)
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

threshold = 0  # e.g., filter for positive latitudes (northern hemisphere) or another appropriate value
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
display(filtered_df.head())

# Normalize the numeric field
norm_field = f"{numeric_field_id}_normalized"
filtered_df[norm_field] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, norm_field]].head())

# Try grouping by a categorical field, e.g. 'cr:Country' or 'Country'
group_field_id = None
for possible in ['cr:Country', 'Country', 'cr:Family', 'Family', 'cr:Genus', 'Genus']:
    if possible in df.columns:
        group_field_id = possible
        break
print(f"Grouping field: {group_field_id}")
if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Mean {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of normalized numeric field
plt.figure(figsize=(8,4))
sns.histplot(filtered_df[norm_field].dropna(), bins=10, kde=True)
plt.title(f"Distribution of normalized {numeric_field_id}")
plt.xlabel(norm_field)
plt.ylabel('Count')
plt.show()

# If latitude and longitude available, plot map
if all(x in df.columns for x in ['Latitude', 'Longitude']):
    plt.figure(figsize=(7,6))
    plt.scatter(df['Longitude'], df['Latitude'], alpha=0.7)
    plt.title('Specimen Locations')
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.show()

## 6. Conclusion
This notebook demonstrated how to use `mlcroissant` to:
- Load and inspect a Croissant-format biodiversity dataset via its schema URL
- Explore its metadata, record sets, and field structure using entity `@id` references
- Extract data into pandas DataFrames for further analysis
- Apply standard EDA, such as filtering, normalization, and grouping
- Visualize numeric fields and specimen geographical distributions

You can further analyze or extend these steps as relevant to your research questions.